# 01 - Extração Bronze: Fake Store API

Extrai dados brutos da [Fake Store API](https://fakestoreapi.com) (produtos)
e grava na camada **Bronze** em formato Iceberg, no bucket
`lakehouse/bronze/data_api`.

- Carga **incremental** via `MERGE INTO` (upsert por `id`)
- Sem nenhuma transformação: espelho fiel da fonte + metadados de carga

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, StructType as StructT
)
from pyspark.sql.functions import current_timestamp, col
import requests
import pandas as pd

## 1. Configuração da SparkSession

Catálogo Iceberg `lakehouse` (JDBC, metastore Postgres) + acesso ao MinIO (S3).

In [ ]:
spark = (
    SparkSession.builder
    .appName("bronze-fakestore-api")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.4")

    # Catálogo Iceberg (JDBC, mesmo metastore do Trino)
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "jdbc")
    .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://postgres:5432/metastore")
    .config("spark.sql.catalog.lakehouse.jdbc.user", "dba_admin")
    .config("spark.sql.catalog.lakehouse.jdbc.password", "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2")
    .config("spark.sql.catalog.lakehouse.jdbc.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")

    # Acesso ao MinIO (S3)
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Extensões Iceberg (necessário para MERGE INTO)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 2. Extração dos dados da API

Endpoint: `GET /products` — retorna produtos com categoria, preço, descrição e avaliação.

In [ ]:
API_URL = "https://fakestoreapi.com/products"

resp = requests.get(API_URL, timeout=30)
resp.raise_for_status()
produtos_json = resp.json()

print(f"Total de produtos extraídos: {len(produtos_json)}")
produtos_json[0]

## 3. Normalização mínima para DataFrame

Achatamos o campo `rating` (objeto aninhado) em colunas simples.
Nenhuma limpeza ou padronização de tipos/strings ainda — isso fica para a camada Prata.

In [ ]:
registros = []
for p in produtos_json:
    rating = p.get("rating", {}) or {}
    registros.append({
        "id": p.get("id"),
        "title": p.get("title"),
        "price": p.get("price"),
        "description": p.get("description"),
        "category": p.get("category"),
        "image": p.get("image"),
        "rating_rate": rating.get("rate"),
        "rating_count": rating.get("count"),
    })

pdf = pd.DataFrame(registros)
pdf.head()

## 4. Criação do DataFrame Spark com schema explícito

In [ ]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("description", StringType(), True),
    StructField("category", StringType(), True),
    StructField("image", StringType(), True),
    StructField("rating_rate", DoubleType(), True),
    StructField("rating_count", IntegerType(), True),
])

df = spark.createDataFrame(pdf, schema=schema)
df = df.withColumn("data_carga", current_timestamp())

df.printSchema()
df.show(5, truncate=40)

## 5. Criar schema e tabela Iceberg na camada Bronze

Localização: `s3a://lakehouse/bronze/data_api`

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS lakehouse.bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.bronze.data_api (
    id           INT,
    title        STRING,
    price        DOUBLE,
    description  STRING,
    category     STRING,
    image        STRING,
    rating_rate  DOUBLE,
    rating_count INT,
    data_carga   TIMESTAMP
)
USING iceberg
LOCATION 's3a://lakehouse/bronze/data_api'
""")

## 6. Carga incremental (MERGE / upsert por `id`)

In [ ]:
df.createOrReplaceTempView("staging_bronze")

spark.sql("""
MERGE INTO lakehouse.bronze.data_api AS target
USING staging_bronze AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
    target.title        = source.title,
    target.price        = source.price,
    target.description  = source.description,
    target.category     = source.category,
    target.image        = source.image,
    target.rating_rate  = source.rating_rate,
    target.rating_count = source.rating_count,
    target.data_carga   = source.data_carga
WHEN NOT MATCHED THEN INSERT (
    id, title, price, description, category, image,
    rating_rate, rating_count, data_carga
)
VALUES (
    source.id, source.title, source.price, source.description, source.category,
    source.image, source.rating_rate, source.rating_count, source.data_carga
)
""")

## 7. Verificação

In [ ]:
spark.sql("SELECT COUNT(*) AS total_registros FROM lakehouse.bronze.data_api").show()
spark.sql("SELECT * FROM lakehouse.bronze.data_api ORDER BY id LIMIT 10").show(truncate=30)

In [ ]:
# Histórico de snapshots (controle de versão Iceberg)
spark.sql("SELECT * FROM lakehouse.bronze.data_api.history").show()

In [ ]:
spark.stop()